In [ ]:
using Plots, DifferentialEquations, NLsolve

In [ ]:
const I0 = 1e-12 # A
const κ = 0.7
const Vdd = 1.8 # V
const UT= 25*1e-3 # V
const C=1e-6 # F
;

\begin{align*}
    \text{M1: }& I_{cmp} = I_{0} e^{\kappa\left(V_{d d}-V_{in}\right) / U_T}\left(1-e^{-\left(V_{d d}-V_1\right) / U_T}\right)\\
    \text{M2: }& I_{1} = I_{0} e^{\kappa V_0 / U_T}\left(1-e^{-V_1 / U_T}\right)\\
    \text{M3: }& I_{lin1} = I_{0} e^{\left(\kappa V_1 - V_{out} )\right/ U_T}\left(1-e^{-\left(V_1-V_{out}\right) / U_T}\right)\\
    \text{M4: }& I_{lin2} = I_{0} e^{\left(\kappa V_{out} - V_{2} \right)/ U_T}\left(1-e^{-\left(V_{out}-V_{2}\right) / U_T}\right)\\
    \text{M5: }& I_{lin3} = I_{0} e^{\kappa V_{lin} / U_T}\left(1-e^{-V_2 / U_T}\right)\\
    \text{K1: }& C_1\dot V_1 = I_{cmp} - I_1 - I_{lin1} \\
    \text{Kout: }& C_{out} \dot V_{out} = I_{lin1} - I_{lin2} \\
    \text{K2: }& C_2\dot V_2 = I_{lin2} - I_{lin3}
\end{align*}

In [ ]:
# Static transistor equations
Icmp(Vin,V1) = I0 * exp(κ*(Vdd-Vin)/UT) * (1 - exp(-(Vdd-V1)/UT))
I1(V0,V1) = I0 * exp(κ*V0/UT) * (1 - exp(-V1/UT))
Ilin1(V1,Vout) = I0 * exp((κ*V1 - Vout)/UT) * (1 - exp(-(V1-Vout)/UT))
Ilin2(Vout,V2) = I0 * exp((κ*Vout - V2)/UT) * (1 - exp(-(Vout-V2)/UT))
Ilin3(Vlin,V2) = I0 * exp(κ*Vlin/UT) * (1 - exp(-V2/UT));

In [ ]:
function Imode_sigmoid!(du,u,p,t)
    
    Vin = p[1]
    Vlin = p[2]
    V0 = p[3]
    
    V1=u[1]
    Vout=u[2]
    V2=u[3]

    du[1]= (Icmp(Vin(t),V1) - I1(V0,V1) - Ilin1(V1,Vout))/C
    du[2]= (Ilin1(V1,Vout) - Ilin2(Vout,V2))/C
    du[3]= (Ilin2(Vout,V2) - Ilin3(Vlin,V2))/C
    
end;

$$V_{P_\text{diode}} = V_{dd} - \frac{U_T}{\kappa} \log \left(\frac{I_{in}}{I_0}\right)$$
$$V_{N_\text{diode}} = \frac{U_T}{\kappa} \log \left(\frac{I_{in}}{I_0}\right)$$

In [ ]:
V_P_diode(I) = Vdd - UT/κ * log(I/I0)
V_N_diode(I) = UT/κ * log(I/I0);

In [ ]:
Ilin = 200e-9 # A
I_0 = 300e-9 # A
Igain = 500e-9 # A

Vlin = V_N_diode(Ilin)
V0 = V_N_diode(I_0)
Vgain = V_N_diode(Igain);

In [ ]:
Tfinal=10.0
tspan=(0.0,Tfinal)

Vin(t) = 1

p=(Vin,Vlin,V0)

x0=[0;0;0]

prob=ODEProblem(Imode_sigmoid!,x0,tspan,p)
sol=solve(prob,Rosenbrock23())
plot(sol)

In [ ]:
# Number of Vin values to simulate
Nreps=100
Iin_range=range(600e-9,1e-9,length=Nreps)
Vin_range=V_P_diode.(Iin_range)
# Vin_range=range(0.1,Vdd,length=Nreps)

# finding an initial guess of IC

function f(x)
    [Icmp(Vin_range[1],x[1]) - I1(V0,x[1]) - Ilin1(x[1],x[2]),
    Ilin1(x[1],x[2]) - Ilin2(x[2],x[3]),
    Ilin2(x[2],x[3]) - Ilin3(Vlin,x[3])]
end

solNL = nlsolve(f, sol[:,end], iterations=convert(Int64,1e6), ftol=1e-9, xtol=1e-6)
#println(solNL.f_converged)
#println(solNL.f_calls)
#solNL.zero

In [ ]:
# Storing all the voltage values in a vector (V1, Vout, V2 are the 3 rows)
Vout_OUT=zeros(3,Nreps)
Vout_OUT[:,1] = solNL.zero # First point is given by NLsolve

Tfinal=100.0
tspan=(0.0,Tfinal)

for i=2:Nreps
    
    Vin(t) = Vin_range[i]
    
    p=(Vin,Vlin,V0)
    
    x0=Vout_OUT[:,i-1]
    
    prob=ODEProblem(Imode_sigmoid!,x0,tspan,p)
    sol=solve(prob,Rosenbrock23());
    Vout_OUT[:,i] = sol[1:3,end]
end
# plot(Vin_range,Vout_OUT[2,:],grid=false,xlabel="Vin",ylabel="Vout",legend=false,title="Vlin=$Vlin, V0=$V0")

\begin{align}
    I_{in} &= I_0 e^{\kappa (V_{dd}-V_{in})/U_T}\\
    I_{out} &= I_0 \frac{e^{\kappa V_{out}/U_T}}{1+e^{\kappa\left(V_{out} - V_{gain}\right)/U_T}}
\end{align}

In [ ]:
Iin(Vin) = I0 * exp(κ*(Vdd-Vin)/UT)
Iout(Vout) = I0 * exp((κ*Vout)/UT) / (1 + exp(κ*(Vout-Vgain)/UT));

In [ ]:
plot(Iin.(collect(Vin_range)),Iout.(Vout_OUT[2,:]),grid=false,xlabel="Iin",ylabel="Iout",legend=false,title="Ilin=$Ilin, I0=$I_0, Igain=$Igain")

The accuracy is poor when the number of points in the solution is low